# 07 · Análisis final: ¿Acacías vive del petróleo?

**Autor:** Carlos Andrés Morales Rojas

**Pregunta:** entre 2012 y 2026, ¿la producción petrolera y las regalías de Acacías (Meta) se tradujeron en mejores condiciones de vida para sus habitantes?

Este notebook **no limpia datos**: usa las tablas que dejaron los notebooks 01 a 06 en `data processed/` y arma la historia con las gráficas clave.

**Diseño de las gráficas:** estilo Apple (tipografía limpia, colores de sistema, barras redondeadas, sin bordes ni ruido). Cada gráfica se guarda en **dos versiones** en `figuras/`:
- `NN_nombre.png`: modo claro
- `NN_nombre_oscuro.png`: modo oscuro

Ambas tienen **fondo transparente**, y el README muestra la que corresponde según el tema del visitante en GitHub.

**Fuentes:** ANH, DNP-SICODIS, DNP-MapaInversiones, DNP-TerriData, SECOP II, DANE. Valores monetarios en **pesos constantes de 2025**.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, Polygon
import numpy as np
from matplotlib.ticker import FuncFormatter

DATA = Path("../data processed")
FIG = Path("../figuras")
FIG.mkdir(exist_ok=True)
ANIO_BASE = 2025

## Sistema de diseño

Colores inspirados en la paleta de sistema de Apple, **validados** para daltonismo y contraste en ambos modos. Cada entidad tiene siempre el mismo color:
- **Acacías:** azul
- **Meta:** naranja
- **Colombia:** verde azulado
- **Otros municipios:** gris (contexto)

**Tipografía:** usa *SF Pro* si está instalada; si no, *Inter* (gratuita y muy parecida, en fonts.google.com/specimen/Inter); si no, *Segoe UI*, que viene con Windows.

In [ ]:
TEMAS = {
    "claro": {
        "sufijo": "", "texto": "#1D1D1F", "texto2": "#6E6E73", "grilla": "#E5E5EA",
        "acacias": "#0071E3", "meta": "#F56300", "colombia": "#00A3A3", "contexto": "#C7C7CC",
        "sombra": "#F2F2F7", "fondo": "#FFFFFF",
    },
    "oscuro": {
        "sufijo": "_oscuro", "texto": "#F5F5F7", "texto2": "#A1A1A6", "grilla": "#2C2C2E",
        "acacias": "#0A84FF", "meta": "#E8650F", "colombia": "#14A3A3", "contexto": "#48484A",
        "sombra": "#1C1C1E", "fondo": "#0D1117",
    },
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["SF Pro Display", "Inter", "Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.size": 11, "figure.dpi": 110, "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
})

def aplicar_tema(fig, axes, t):
    fig.patch.set_alpha(0)
    for ax in (axes if hasattr(axes, "__iter__") else [axes]):
        ax.set_facecolor("none")
        ax.tick_params(colors=t["texto2"], labelsize=10)
        ax.spines["bottom"].set_color(t["grilla"])
        ax.title.set_color(t["texto"])
        ax.xaxis.label.set_color(t["texto2"])
        ax.yaxis.label.set_color(t["texto2"])

def encabezado(fig, titulo, subtitulo, t, espacio=0.0):
    """Título y subtítulo alineados con el borde izquierdo de las gráficas.
    `espacio` (en pulgadas) deja lugar extra si los paneles tienen su propio título."""
    cajas = [ax.get_position() for ax in fig.axes if ax.get_visible()]
    x0 = min(c.x0 for c in cajas)
    y1 = max(c.y1 for c in cajas)
    alto = fig.get_size_inches()[1]
    fig.text(x0, y1 + (0.95 + espacio) / alto, titulo, fontsize=19, fontweight="bold", color=t["texto"], ha="left", va="bottom")
    fig.text(x0, y1 + (0.62 + espacio) / alto, subtitulo, fontsize=12, color=t["texto2"], ha="left", va="bottom")

def pie(fig, texto, t):
    x0 = min(ax.get_position().x0 for ax in fig.axes if ax.get_visible())
    fig.text(x0, -0.02, texto, fontsize=9, color=t["texto2"], ha="left", va="top")

def barras(ax, x, largo, grosor, color, horizontal=False, base=0, radio=0.3):
    """Barras con el extremo redondeado (estilo Apple) y la base recta.
    Fija los límites de los ejes antes de llamarla: el redondeo depende de la escala."""
    (x1, y1), (x2, y2) = ax.transData.transform([(0, 0), (1, 1)])
    px, py = abs(x2 - x1), abs(y2 - y1)          # píxeles por unidad de x y de y
    ang = np.linspace(0, np.pi / 2, 12)
    for xi, li in zip(x, largo):
        if pd.isna(li) or li <= 0:
            continue
        r_pix = min(radio * grosor * (py if horizontal else px), li * (px if horizontal else py))
        if horizontal:
            rx, ry = r_pix / px, min(r_pix / py, grosor / 2)
            x0, y0, y1_ = base, xi - grosor / 2, xi + grosor / 2
            xe = base + li
            pts = [(x0, y0), (xe - rx, y0)]
            pts += [(xe - rx + rx * np.sin(a), y0 + ry - ry * np.cos(a)) for a in ang]
            pts += [(xe - rx + rx * np.cos(a), y1_ - ry + ry * np.sin(a)) for a in ang]
            pts += [(x0, y1_)]
        else:
            rx, ry = min(r_pix / px, grosor / 2), r_pix / py
            x0, x1_, ye = xi - grosor / 2, xi + grosor / 2, base + li
            pts = [(x0, base), (x0, ye - ry)]
            pts += [(x0 + rx - rx * np.cos(a), ye - ry + ry * np.sin(a)) for a in ang]
            pts += [(x1_ - rx + rx * np.sin(a), ye - ry + ry * np.cos(a)) for a in ang]
            pts += [(x1_, base)]
        ax.add_patch(Polygon(pts, closed=True, color=color, lw=0))

def guardar_y_mostrar(dibujar, nombre):
    for tema, t in TEMAS.items():
        plt.rcParams["grid.color"] = t["grilla"]
        fig = dibujar(t)
        fig.savefig(FIG / f"{nombre}{t['sufijo']}.png", bbox_inches="tight", transparent=True, pad_inches=0.25)
        if tema == "claro":
            plt.show()
        else:
            plt.close(fig)

fmt_puntos = FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", "."))
coma = lambda v, d=1: f"{v:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")

## 0. Cifras clave

Una tarjeta-resumen con los tres números que cuentan la historia, para la portada del README.

In [ ]:
anh = pd.read_csv(DATA / "anh_municipios_anual.csv")
ac = anh[anh["municipio"] == "ACACIAS"].set_index("anio")
ac["bpd"] = ac["petroleo_bls"] / (ac["meses_reportados"] * 365 / 12)

gvr = pd.read_csv(DATA / "acacias_generado_vs_recibido.csv").set_index("anio").loc[2013:2025]
gen = gvr[f"regalias_generadas_anh_const{ANIO_BASE}"] / 1e9
rec = gvr[f"asignaciones_directas_recibidas_const{ANIO_BASE}"] / 1e9

censo = pd.read_csv(DATA / "terridata_pobreza_censos.csv")
ipm = (censo[censo["indicador"] == "indice de pobreza multidimensional - IPM (Censo)"]
         .set_index("lugar")[["2005", "2018"]].astype(float))

CIFRAS = [
    (f"{ac.loc[2025, 'bpd'] / 1000:.0f} mil", "barriles de petróleo al día", "2.º productor del Meta · 2025"),
    (f"${100 * rec.sum() / gen.sum():.0f}", "de cada $100 en regalías", "llegan a la alcaldía · 2013-2025"),
    (f"{ipm.loc['Acacías', '2005']:.0f}% → {ipm.loc['Acacías', '2018']:.0f}%", "pobreza multidimensional", "censos 2005 y 2018"),
]

def dibujar_cifras(t):
    fig, ax = plt.subplots(figsize=(13, 2.8))
    ax.set_axis_off()
    ax.set_xlim(0, 3); ax.set_ylim(0, 1)
    for i, (num, etiqueta, nota) in enumerate(CIFRAS):
        ax.add_patch(FancyBboxPatch((i + 0.04, 0.04), 0.92, 0.92, boxstyle="round,pad=0,rounding_size=0.08",
                                    mutation_aspect=13 / 2.8 / 3, color=t["sombra"], lw=0))
        ax.text(i + 0.12, 0.62, num, fontsize=30, fontweight="bold", color=t["acacias"], va="center")
        ax.text(i + 0.12, 0.34, etiqueta, fontsize=13, color=t["texto"], va="center")
        ax.text(i + 0.12, 0.18, nota, fontsize=10, color=t["texto2"], va="center")
    fig.patch.set_alpha(0)
    return fig

guardar_y_mostrar(dibujar_cifras, "00_cifras_clave")

## 1. Acacías es una potencia petrolera

In [ ]:
def dibujar_produccion(t):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_xlim(2011.3, 2026.7); ax.set_ylim(0, 125_000)
    barras(ax, ac.index, ac["bpd"], 0.62, t["acacias"])
    ax.yaxis.set_major_formatter(fmt_puntos)
    ax.set_xticks(ac.index)
    ax.annotate(coma(ac.loc[2025, "bpd"], 0), (2025, ac.loc[2025, "bpd"]), textcoords="offset points",
                xytext=(0, 6), ha="center", fontsize=10, fontweight="bold", color=t["texto"])
    aplicar_tema(fig, ax, t)
    encabezado(fig, "Cerca de 100.000 barriles al día", "Producción de petróleo de Acacías, barriles por día", t)
    pie(fig, "Fuente: ANH, producción fiscalizada. 2026 = enero a junio.", t)
    return fig

guardar_y_mostrar(dibujar_produccion, "01_produccion_acacias")

## 2. Genera mucho, recibe poco

La producción de Acacías genera cientos de miles de millones de pesos al año en regalías. Con el Sistema General de Regalías, la mayor parte se reparte entre fondos regionales, el departamento y el ahorro. A la **alcaldía** le llegan solo las **Asignaciones Directas**.

In [ ]:
por100 = 100 * rec / gen

def dibujar_gvr(t):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw={"width_ratios": [1.7, 1], "wspace": 0.18})
    x = gvr.index
    ax[0].set_xlim(2012.4, 2025.6); ax[0].set_ylim(0, 1300)
    barras(ax[0], x - 0.19, gen, 0.36, t["meta"])
    barras(ax[0], x + 0.19, rec, 0.36, t["acacias"])
    ax[0].set_xticks(x); ax[0].tick_params(axis="x", rotation=90)
    ax[0].set_title("Miles de millones de pesos de 2025", loc="left", fontsize=11, color=t["texto2"], pad=10)
    ax[0].text(2022 - 0.19, gen[2022] + 25, "Generadas", ha="center", fontsize=10, color=t["texto"], fontweight="bold")
    ax[0].text(2023 + 0.19, rec[2023] + 25, "Recibidas", ha="left", fontsize=10, color=t["texto"], fontweight="bold")
    ax[0].yaxis.set_major_formatter(fmt_puntos)
    from matplotlib.patches import Patch
    ax[0].legend(handles=[Patch(color=t["meta"], label="Generadas por el petróleo de Acacías (ANH)"),
                          Patch(color=t["acacias"], label="Recibidas por la alcaldía (SICODIS)")],
                 loc="upper left", fontsize=10, labelcolor=t["texto"])

    ax[1].set_xlim(2012.4, 2025.6); ax[1].set_ylim(0, 10.5)
    barras(ax[1], x, por100, 0.62, t["acacias"])
    ax[1].set_xticks(x); ax[1].tick_params(axis="x", rotation=90)
    ax[1].set_title("Pesos que recibe por cada $100 generados", loc="left", fontsize=11, color=t["texto2"], pad=10)
    for anio in [2013, 2020, 2023, 2025]:
        ax[1].annotate(f"${coma(por100[anio])}", (anio, por100[anio]), textcoords="offset points",
                       xytext=(0, 5), ha="center", fontsize=10, fontweight="bold", color=t["texto"])
    aplicar_tema(fig, ax, t)
    encabezado(fig, "Genera mucho, recibe poco",
               "Regalías que produce el petróleo de Acacías frente a lo que recibe la alcaldía", t, espacio=0.35)
    pie(fig, f"Fuentes: ANH y DNP-SICODIS. Pesos constantes de {ANIO_BASE} (IPC DANE).", t)
    return fig

guardar_y_mostrar(dibujar_gvr, "02_generado_vs_recibido")
print(f"2013-2025: generadas {gen.sum():,.0f} mm, recibidas {rec.sum():,.0f} mm -> ${100 * rec.sum() / gen.sum():.1f} por cada $100")

**Hallazgo:** la alcaldía recibe entre **$2 y $9 por cada $100** de regalías que produce su petróleo. Desde 2021, con el nuevo régimen del SGR (Ley 2056 de 2020), la porción subió, pero sigue siendo menos de una décima parte.

## 3. ¿En qué se invirtió?

In [ ]:
sect = pd.read_csv(DATA / "proyectos_sgr_por_sector.csv").set_index("sector")["valor_mm"].sort_values()
nombres = {"Vivienda, Ciudad Y Territorio": "Vivienda, agua y alcantarillado",
           "Tecnologías De La Información Y Las Comunicaciones": "Tecnología (TIC)",
           "Comercio, Industria Y Turismo": "Comercio y turismo", "Ambiente Y Desarrollo Sostenible": "Ambiente",
           "Deporte Y Recreación": "Deporte y recreación", "Minas Y Energía": "Minas y energía",
           "Gobierno Territorial": "Gobierno (edificios y gestión)"}
sect.index = [nombres.get(s, s) for s in sect.index]
pct = 100 * sect / sect.sum()

def dibujar_sectores(t):
    fig, ax = plt.subplots(figsize=(12, 5.6))
    y = range(len(sect))
    ax.set_ylim(-0.6, len(sect) - 0.4); ax.set_xlim(0, sect.max() * 1.12)
    destacados = {"Transporte", "Vivienda, agua y alcantarillado"}
    for yi, (nombre, v) in zip(y, sect.items()):
        barras(ax, [yi], [v], 0.62, t["acacias"] if nombre in destacados else t["contexto"], horizontal=True)
        ax.text(v + sect.max() * 0.012, yi, f"{pct[nombre]:.0f}%", va="center", fontsize=10,
                color=t["texto"], fontweight="bold" if nombre in destacados else "normal")
    ax.set_yticks(list(y)); ax.set_yticklabels(sect.index)
    ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.xaxis.set_major_formatter(fmt_puntos)
    ax.set_xlabel(f"Miles de millones de pesos de {ANIO_BASE}")
    aplicar_tema(fig, ax, t)
    ax.tick_params(axis="y", colors=t["texto"])
    encabezado(fig, "Vías, vivienda y agua: casi 7 de cada 10 pesos",
               "Proyectos financiados con regalías por la Alcaldía de Acacías, por sector", t)
    pie(fig, "Fuente: DNP-MapaInversiones. Proyectos SGR terminados, en ejecución o viables.", t)
    return fig

guardar_y_mostrar(dibujar_sectores, "03_inversion_por_sector")

## 4. ¿Mejoró la vida de la gente?

Comparamos el **último dato disponible** de Acacías con el del Meta y el de Colombia.

In [ ]:
panel = pd.read_csv(DATA / "terridata_panel_indicadores.csv")
elegidos = ["Cobertura acueducto (%)", "Cobertura alcantarillado (%)", "Internet fijo (%)",
            "Saber 11 Matemáticas (puntos)", "Mortalidad infantil (x 1.000)", "Homicidios (x 100.000 hab.)",
            "Tránsito a educación superior (%)", "Deserción escolar oficial (%)"]
titulos = {"Cobertura acueducto (%)": "Acueducto", "Cobertura alcantarillado (%)": "Alcantarillado",
           "Internet fijo (%)": "Internet fijo", "Saber 11 Matemáticas (puntos)": "Saber 11 Matemáticas",
           "Mortalidad infantil (x 1.000)": "Mortalidad infantil", "Homicidios (x 100.000 hab.)": "Homicidios",
           "Tránsito a educación superior (%)": "Paso a la universidad", "Deserción escolar oficial (%)": "Deserción escolar"}
unidades = {"Cobertura acueducto (%)": "% cobertura", "Cobertura alcantarillado (%)": "% cobertura",
            "Internet fijo (%)": "% penetración", "Saber 11 Matemáticas (puntos)": "puntos",
            "Mortalidad infantil (x 1.000)": "por 1.000 nacidos vivos", "Homicidios (x 100.000 hab.)": "por 100.000 hab.",
            "Tránsito a educación superior (%)": "% de bachilleres", "Deserción escolar oficial (%)": "% colegios oficiales"}
lugares = ["Acacías", "Meta", "Colombia"]

ult = (panel[panel["nombre"].isin(elegidos) & panel["lugar"].isin(lugares) & panel["valor"].notna() & (panel["anio"] <= 2025)]
         .sort_values("anio").groupby(["nombre", "lugar"]).last().reset_index())
mejor = panel.drop_duplicates("nombre").set_index("nombre")["mejor"]

def dibujar_indicadores(t):
    colores = [t["acacias"], t["meta"], t["colombia"]]
    fig, axes = plt.subplots(2, 4, figsize=(16, 7.8), gridspec_kw={"hspace": 0.75, "wspace": 0.28})
    for ax, nombre in zip(axes.flat, elegidos):
        sub = ult[ult["nombre"] == nombre].set_index("lugar").reindex(lugares)
        top = sub["valor"].max() * 1.22
        ax.set_xlim(-0.6, 2.6); ax.set_ylim(0, top)
        for i, (v, c) in enumerate(zip(sub["valor"], colores)):
            barras(ax, [i], [v], 0.62, c)
            ax.text(i, v + top * 0.02, coma(v), ha="center", va="bottom", fontsize=10,
                    fontweight="bold" if i == 0 else "normal", color=t["texto"])
        ax.set_xticks(range(3)); ax.set_xticklabels(lugares)
        ax.set_yticks([])
        flecha = "↑ mejor" if mejor[nombre] == "sube" else "↓ mejor"
        ax.set_title(f"{titulos[nombre]}", loc="left", fontsize=12, fontweight="bold", pad=22)
        ax.text(0, 1.04, f"{unidades[nombre]} · {flecha} · {int(sub.loc['Acacías', 'anio'])}",
                transform=ax.transAxes, fontsize=9, color=t["texto2"])
        ax.grid(False)
    aplicar_tema(fig, axes.flat, t)
    encabezado(fig, "Mejor que el promedio, salvo en la universidad",
               "Acacías frente al Meta y a Colombia, último dato disponible", t, espacio=0.45)
    pie(fig, "Fuente: DNP-TerriData.", t)
    return fig

guardar_y_mostrar(dibujar_indicadores, "04_indicadores_sociales")

In [ ]:
ipm_orden = ipm.sort_values("2018")

def dibujar_pobreza(t):
    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.set_xlim(8, 88); ax.set_ylim(-0.7, len(ipm_orden) - 0.3)
    for i, (lugar, r) in enumerate(ipm_orden.iterrows()):
        c = t["acacias"] if lugar == "Acacías" else t["contexto"]
        ax.plot([r["2005"], r["2018"]], [i, i], color=c, lw=4 if lugar == "Acacías" else 2.5,
                solid_capstyle="round", zorder=1)
        ax.scatter(r["2005"], i, s=90, facecolor=t["fondo"], edgecolor=c, lw=2.5, zorder=2)
        ax.scatter(r["2018"], i, s=90, color=c, zorder=2)
        peso = "bold" if lugar == "Acacías" else "normal"
        ax.text(r["2018"] - 2, i, coma(r["2018"]), ha="right", va="center", fontsize=10, color=t["texto"], fontweight=peso)
        ax.text(r["2005"] + 2, i, coma(r["2005"]), ha="left", va="center", fontsize=10, color=t["texto2"])
    ax.set_yticks(range(len(ipm_orden))); ax.set_yticklabels(ipm_orden.index)
    ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.set_xlabel("% de personas en pobreza multidimensional   ○ 2005   ● 2018")
    aplicar_tema(fig, ax, t)
    ax.tick_params(axis="y", colors=t["texto"])
    encabezado(fig, "La pobreza cayó a menos de la mitad",
               "Índice de pobreza multidimensional, censos 2005 y 2018", t)
    pie(fig, "Fuente: DANE, censos 2005 y 2018, vía DNP-TerriData.", t)
    return fig

guardar_y_mostrar(dibujar_pobreza, "05_pobreza_censos")

## 5. La contratación sigue el calendario electoral

In [ ]:
mensual = pd.read_csv(DATA / "secop_alcaldia_mensual.csv", parse_dates=["mes"]).set_index("mes").fillna(0)
directa = mensual.get("Directa", 0)
otras = mensual.drop(columns="Directa", errors="ignore").sum(axis=1)

def dibujar_garantias(t):
    fig, ax = plt.subplots(figsize=(15, 5.2))
    for ini, fin in [("2022-01-29", "2022-06-19"), ("2026-01-31", "2026-06-21")]:
        ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fin), color=t["sombra"], zorder=0, lw=0)
        ax.text(pd.Timestamp(ini) + pd.Timedelta(days=10), 600, "Ley de\nGarantías", fontsize=10,
                color=t["texto2"], va="top")
    ax.bar(mensual.index, directa, width=22, color=t["acacias"], label="Contratación directa")
    ax.bar(mensual.index, otras, bottom=directa, width=22, color=t["meta"], label="Otras modalidades")
    ax.annotate("Enero 2026\n618 contratos", (pd.Timestamp("2026-01-01"), 618), xytext=(-120, -12),
                textcoords="offset points", fontsize=10, fontweight="bold", color=t["texto"],
                arrowprops=dict(arrowstyle="-", color=t["texto2"], lw=1))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlim(pd.Timestamp("2021-01-01"), pd.Timestamp("2026-12-31")); ax.set_ylim(0, 660)
    ax.legend(loc="upper left", fontsize=10, labelcolor=t["texto"])
    aplicar_tema(fig, ax, t)
    encabezado(fig, "Enero se dispara antes de las elecciones",
               "Contratos firmados por mes por la Alcaldía de Acacías", t)
    pie(fig, "Fuente: SECOP II. Zonas sombreadas: prohibición de contratación directa antes de elecciones presidenciales (Ley 996 de 2005).", t)
    return fig

guardar_y_mostrar(dibujar_garantias, "06_ley_de_garantias")

**Ojo con la lectura:** cada año arranca con un pico de contratos en enero y febrero, porque se renuevan los contratos de prestación de servicios. En los años normales (2023 a 2025) ese arranque suma entre 226 y 353 contratos. Con la Ley de Garantías la contratación directa queda prohibida desde finales de enero, así que todo se adelanta a enero: 349 contratos en enero de 2022 y **618 en enero de 2026**, casi el doble de un arranque normal.

## 6. Conclusiones

1. **Acacías es un gigante petrolero:** produce alrededor de 100.000 barriles diarios, y las actividades primarias son cerca del 80% de su economía.
2. **Pero la alcaldía recibe una fracción pequeña de las regalías que genera:** entre $2 y $9 por cada $100. El resto se reparte en el sistema nacional.
3. **Lo que llega se invierte sobre todo en cemento:** vías, vivienda y acueducto/alcantarillado se llevan casi el 70% de los proyectos con regalías.
4. **Acacías está mejor que el promedio** del Meta y de Colombia en servicios públicos, pruebas Saber, internet y seguridad, y su pobreza multidimensional cayó a menos de la mitad entre 2005 y 2018.
5. **Pero el salto de regalías desde 2021 no se nota claramente en los indicadores**, y el acceso a la educación superior sigue rezagado.
6. **La contratación responde al calendario electoral:** antes de la Ley de Garantías toda la contratación se adelanta a enero (618 contratos en enero de 2026), y una "nómina paralela" de cientos de contratistas crece cada año.

## Limitaciones

- **Correlación no es causalidad:** que un indicador mejore mientras llegan regalías no prueba que las regalías lo causaron.
- **Regalías "recibidas" = solo Asignaciones Directas.** Acacías también se beneficia de proyectos regionales o departamentales; la inversión de la Gobernación dentro del municipio no se cuantificó.
- **Cambios de metodología** en algunos indicadores (vacunación 2021, mortalidad infantil 2017) impiden leer tendencias largas en ellos.
- **SECOP II cubre la alcaldía solo desde 2021.**
- **Avances físicos en 0%** en MapaInversiones parecen falta de reporte (el cruce con SECOP muestra contratos ejecutados y prorrogados), no obras abandonadas.

## Próximos pasos

- Comparar con municipios **similares sin petróleo** (por ejemplo con *matching*) para estimar el efecto real de las regalías.
- Agregar la inversión de la Gobernación del Meta en Acacías.